In [0]:
from pyspark.sql.functions import col
# 1. Join Silver tables to create the master Fact Table
# We use silver_orders as the base, joining leads for ID/metrics, and inventory for product/date details
df_gold_fact = spark.table("my_project_db.silver_orders").alias("o") \
    .join(spark.table("my_project_db.silver_leads").alias("l"), "lead_id", "left") \
    .join(spark.table("my_project_db.silver_inventory").alias("i"), "invt_id", "left") \
    .select(
        "o.order_id",
        "o.lead_id",
        "l.invt_id",
        col("l.quantity").alias("order_qty"),
        col("i.quantity").alias("inventory_qty"),
        "o.order_value",
        "i.purchase_cost",
        "l.lead_date",       # When the lead started
        "o.order_in_date",   # When order was placed
        "o.order_out_date",  # When order was fulfilled
        "o.reason_name",     # Include reason for delay
        "i.warranty_date"    # Purchase/Warranty start date
    )

# 2. Create the Dimension Tables (The points of the star)
df_gold_dim_leads = spark.table("my_project_db.silver_leads") \
    .select("lead_id", "lead_date", "username", "region", "customer_type", "source_name", "status_name") \
    .dropDuplicates(["lead_id"])

df_gold_dim_products = spark.table("my_project_db.silver_inventory") \
    .select("invt_id", "product_sku", "division", "status_name")

# 3. Write them as gold tables
# Use this for all tables where you've added or removed columns
df_gold_fact.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("my_project_db.gold_sales_fact")

df_gold_dim_leads.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("my_project_db.gold_dim_leads")

df_gold_dim_products.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("my_project_db.gold_dim_products")

print("Gold Layer Star Schema updated successfully!")

In [0]:
from pyspark.sql.functions import col, year, month, quarter, dayofmonth

# 1. Collect all dates from your silver tables (including leads) into one unique list
dates_df = spark.table("my_project_db.silver_orders").select(col("order_in_date").alias("date")) \
    .union(spark.table("my_project_db.silver_orders").select(col("order_out_date").alias("date"))) \
    .union(spark.table("my_project_db.silver_inventory").select(col("warranty_date").alias("date"))) \
    .union(spark.table("my_project_db.silver_leads").select(col("lead_date").alias("date"))) \
    .distinct() \
    .na.drop()

# 2. Enrich the date dimension with helpful attributes
df_gold_dim_date = dates_df \
    .withColumn("year", year("date")) \
    .withColumn("month", month("date")) \
    .withColumn("quarter", quarter("date")) \
    .withColumn("day", dayofmonth("date")) \
    .orderBy("date")

# 3. Save as the gold dimension table
df_gold_dim_date.write.format("delta").mode("overwrite").saveAsTable("my_project_db.gold_dim_date")

print("gold_dim_date table updated and saved successfully!")